**Ollama**

In [ ]:
# Instala o Ollama no Colab
!apt-get update -qq > /dev/null
!apt-get install zstd -y > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Inicia o servidor em background
subprocess.Popen(["ollama", "serve"])
time.sleep(4)

# Baixa o modelo Llama 3.2 1B
!ollama pull llama3.2:1b

# Instala a biblioteca Python do Ollama
!pip install ollama -q

print("\n✅ Ollama pronto!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


✅ Ollama pronto!


**Dados simulados da missão**

In [ ]:
import random

# ──────────────────────────────────────────────
# DADOS SIMULADOS DA MISSÃO SOLARIS-1
# ──────────────────────────────────────────────

dados_missao = {
    "temperatura": {
        "modulo_principal":  random.randint(60, 110),   # °C
        "modulo_propulsao":  random.randint(70, 130),   # °C
        "modulo_suporte":    random.randint(50, 95),    # °C
    },
    "energia": {
        "bateria":           random.randint(10, 100),   # %
        "geracao_solar":     random.randint(30, 100),   # %
        "consumo_atual":     random.randint(200, 600),  # W
    },
    "comunicacao": {
        "status_sinal":      random.choice(["ESTÁVEL", "INSTÁVEL", "PERDIDO"]),
        "latencia_ms":       random.randint(80, 2000),
        "pacotes_perdidos":  random.randint(0, 40),     # %
    },
    "modulos": {
        "MÓDULO_PRINCIPAL":  random.choice(["OPERACIONAL", "OPERACIONAL", "FALHA"]),
        "MÓDULO_PROPULSÃO":  random.choice(["OPERACIONAL", "STAND-BY", "FALHA"]),
        "MÓDULO_SUPORTE":    random.choice(["OPERACIONAL", "OPERACIONAL", "STAND-BY"]),
        "MÓDULO_CIENTÍFICO": random.choice(["OPERACIONAL", "STAND-BY"]),
    },
}

print("✅ Dados simulados da missão Solaris-1 carregados!")
print(f"     Temp. módulo principal : {dados_missao['temperatura']['modulo_principal']}°C")
print(f"     Bateria                : {dados_missao['energia']['bateria']}%")
print(f"     Status do sinal        : {dados_missao['comunicacao']['status_sinal']}")

✅ Dados simulados da missão Solaris-1 carregados!
     Temp. módulo principal : 100°C
     Bateria                : 26%
     Status do sinal        : PERDIDO


**Funções de monitoramento**

In [ ]:
# FUNÇÕES DE MONITORAMENTO


def status_temperatura() -> str:
    t = dados_missao["temperatura"]
    alertas = []
    if t["modulo_principal"] > 90:
        alertas.append(f"⚠️  ALERTA: Módulo Principal em {t['modulo_principal']}°C (limite: 90°C)")
    if t["modulo_propulsao"] > 110:
        alertas.append(f"🚨 CRÍTICO: Módulo Propulsão em {t['modulo_propulsao']}°C (limite: 110°C)")
    if t["modulo_suporte"] > 85:
        alertas.append(f"⚠️  ALERTA: Módulo Suporte em {t['modulo_suporte']}°C (limite: 85°C)")

    resultado = (
        f"🌡️  Temperaturas dos módulos:\n"
        f"   • Principal  : {t['modulo_principal']}°C\n"
        f"   • Propulsão  : {t['modulo_propulsao']}°C\n"
        f"   • Suporte    : {t['modulo_suporte']}°C\n"
    )
    if alertas:
        resultado += "\n" + "\n".join(alertas)
    else:
        resultado += "\n✅ Todas as temperaturas dentro do normal."
    return resultado


def status_energia() -> str:
    e = dados_missao["energia"]
    alertas = []

    if e["bateria"] < 20:
        alertas.append("🚨 CRÍTICO: Bateria abaixo de 20% — ativando MODO ECONOMIA DE ENERGIA")
    elif e["bateria"] < 40:
        alertas.append("⚠️  ALERTA: Bateria abaixo de 40% — recomendado reduzir consumo")

    if e["geracao_solar"] < 40:
        alertas.append("⚠️  ALERTA: Geração solar abaixo de 40% — verificar painéis")

    resultado = (
        f"⚡ Status energético:\n"
        f"   • Bateria          : {e['bateria']}%\n"
        f"   • Geração solar    : {e['geracao_solar']}%\n"
        f"   • Consumo atual    : {e['consumo_atual']}W\n"
    )
    if alertas:
        resultado += "\n" + "\n".join(alertas)
    else:
        resultado += "\n✅ Sistema energético operando normalmente."
    return resultado


def status_comunicacao() -> str:
    c = dados_missao["comunicacao"]
    alertas = []

    if c["status_sinal"] == "PERDIDO":
        alertas.append("🚨 CRÍTICO: Sinal perdido — protocolo de emergência iniciado")
    elif c["status_sinal"] == "INSTÁVEL":
        alertas.append("⚠️  ALERTA: Sinal instável — monitorar continuamente")

    if c["pacotes_perdidos"] > 20:
        alertas.append(f"⚠️  ALERTA: {c['pacotes_perdidos']}% de pacotes perdidos — degradação de comunicação")

    resultado = (
        f"📡 Status de comunicação:\n"
        f"   • Sinal           : {c['status_sinal']}\n"
        f"   • Latência        : {c['latencia_ms']} ms\n"
        f"   • Pacotes perdidos: {c['pacotes_perdidos']}%\n"
    )
    if alertas:
        resultado += "\n" + "\n".join(alertas)
    else:
        resultado += "\n✅ Comunicação operando normalmente."
    return resultado


def status_modulos() -> str:
    def icone(s):
        if s == "OPERACIONAL": return "🟢"
        if s == "STAND-BY":    return "🟡"
        return "🔴"

    linhas = "\n".join(
        f"   {icone(s)} {mod}: {s}"
        for mod, s in dados_missao["modulos"].items()
    )
    falhas = [m for m, s in dados_missao["modulos"].items() if s == "FALHA"]
    resultado = f"🛰️  Status dos módulos da missão:\n{linhas}\n"
    if falhas:
        resultado += f"\n🚨 FALHA DETECTADA em: {', '.join(falhas)}"
    else:
        resultado += "\n✅ Todos os módulos respondendo."
    return resultado


def relatorio_completo() -> str:
    return (
        status_temperatura() + "\n\n" +
        status_energia()     + "\n\n" +
        status_comunicacao() + "\n\n" +
        status_modulos()
    )


# Mapa de ferramentas
FERRAMENTAS_DISPONIVEIS = {
    "status_temperatura":  status_temperatura,
    "status_energia":      status_energia,
    "status_comunicacao":  status_comunicacao,
    "status_modulos":      status_modulos,
    "relatorio_completo":  relatorio_completo,
}

print("✅ Funções de monitoramento carregadas!")

✅ Funções de monitoramento carregadas!


**IA com Llama**

In [ ]:
import ollama

# SYSTEM PROMPT — CONTEXTO DA MISSÃO ESPACIAL


SYSTEM_PROMPT = """
Você é Polaris (Artificial Response & Intelligence Assistant), sistema de controle da missão espacial Solaris-1.
Sua função é monitorar as condições operacionais da nave e apoiar a equipe de missão em tempo real.

Você tem acesso às seguintes ferramentas de monitoramento:
  - status_temperatura   : verifica temperaturas dos módulos
  - status_energia       : verifica bateria, geração solar e consumo
  - status_comunicacao   : verifica qualidade do sinal e latência
  - status_modulos       : verifica o estado operacional de cada módulo
  - relatorio_completo   : gera relatório geral de todos os sistemas

Regras de operação:
  1. Quando o usuário perguntar sobre algum sistema, use SEMPRE a ferramenta correspondente para buscar os dados reais.
  2. Identifique e comunique claramente qualquer alerta ou situação crítica.
  3. Se a bateria estiver abaixo de 20%, recomende imediatamente o Modo Economia de Energia.
  4. Se temperatura crítica for detectada, recomende redução de carga no módulo afetado.
  5. Se o sinal for perdido, acione o protocolo de comunicação de emergência.
  6. Responda de forma clara, técnica e objetiva, em português.
  7. Responda APENAS perguntas relacionadas à missão Solaris-1 e seus sistemas.
"""



# DETECÇÃO DE INTENÇÃO ------------------------------------------------------------------


def detectar_ferramenta(mensagem: str):
    msg = mensagem.lower()

    if any(p in msg for p in ["temperatura", "calor", "quente", "frio", "grau", "°c"]):
        return "status_temperatura"
    if any(p in msg for p in ["energia", "bateria", "solar", "consumo", "potência"]):
        return "status_energia"
    if any(p in msg for p in ["comunicação", "sinal", "comunicacao", "latência", "pacote"]):
        return "status_comunicacao"
    if any(p in msg for p in ["módulo", "modulo", "propulsão", "propulsao", "suporte", "status"]):
        return "status_modulos"
    if any(p in msg for p in ["tudo", "geral", "relatório", "relatorio", "completo", "missão", "resumo"]):
        return "relatorio_completo"
    return None


# PROCESSAMENTO DA MENSAGEM COM LLAMA ------------------------------------------------------


def processar_mensagem(historico: list) -> str:
    ultima_mensagem = historico[-1]["content"] if historico else ""
    ferramenta = detectar_ferramenta(ultima_mensagem)

    historico_envio = list(historico)

    if ferramenta:
        dados_reais = FERRAMENTAS_DISPONIVEIS[ferramenta]()
        historico_envio[-1] = {
            "role": "user",
            "content": (
                f"[DADOS DO SISTEMA — {ferramenta.upper()}]\n"
                f"{dados_reais}\n\n"
                f"Com base nesses dados, responda: {ultima_mensagem}"
            )
        }

    response = ollama.chat(
        model="llama3.2:1b",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + historico_envio,
    )

    return response["message"]["content"]


print("✅ Sistema de IA Polaris carregado!")

✅ Sistema de IA Polaris carregado!


**Chatbot**

In [ ]:
# PAINEL DE STATUS INICIAL


def exibir_status_inicial():
    e = dados_missao["energia"]
    t = dados_missao["temperatura"]
    c = dados_missao["comunicacao"]

    def cor_bateria(v):
        if v < 20: return "🔴"
        if v < 40: return "🟡"
        return "🟢"

    def cor_temp(v, limite):
        return "🔴" if v > limite else ("🟡" if v > limite * 0.85 else "🟢")

    def cor_sinal(s):
        return {"ESTÁVEL": "🟢", "INSTÁVEL": "🟡", "PERDIDO": "🔴"}.get(s, "⚪")

    print("=" * 55)
    print("  🚀  MISSION CONTROL AI — SOLARIS-1")
    print("  POLARIS — Artificial Response & Intelligence Assistant")
    print("=" * 55)
    print(f"  {cor_bateria(e['bateria'])} Bateria           : {e['bateria']}%")
    print(f"  ☀️  Geração Solar      : {e['geracao_solar']}%")
    print(f"  {cor_temp(t['modulo_principal'], 90)} Temp. Principal    : {t['modulo_principal']}°C")
    print(f"  {cor_temp(t['modulo_propulsao'], 110)} Temp. Propulsão    : {t['modulo_propulsao']}°C")
    print(f"  {cor_sinal(c['status_sinal'])} Sinal              : {c['status_sinal']}")
    print("-" * 55)

    alertas = []
    if e["bateria"] < 20:
        alertas.append("🚨 CRÍTICO: Bateria < 20% — MODO ECONOMIA ATIVADO")
    if t["modulo_propulsao"] > 110:
        alertas.append("🚨 CRÍTICO: Temperatura de propulsão excedida")
    if c["status_sinal"] == "PERDIDO":
        alertas.append("🚨 CRÍTICO: Sinal perdido — PROTOCOLO DE EMERGÊNCIA")

    if alertas:
        for a in alertas:
            print(f"  {a}")
        print("-" * 55)

    print("  Digite 'sair' para encerrar a sessão.")
    print("=" * 55)
    print()


# ──────────────────────────────────────────────
# LOOP PRINCIPAL — CHATBOT Polaris


def iniciar_chatbot():
    exibir_status_inicial()

    historico = []
    DESPEDIDAS = ("sair", "exit", "tchau", "até", "ate", "bye", "encerrar", "fim")

    while True:
        try:
            entrada = input("Você: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\nPOLARIS: Sessão encerrada. Boa missão, equipe! 🚀")
            break

        if not entrada:
            continue

        if any(d in entrada.lower() for d in DESPEDIDAS):
            print("\nPOLARIS: Sessão encerrada. Solaris-1, até a próxima janela de comunicação. 🚀\n")
            break

        historico.append({"role": "user", "content": entrada})
        resposta = processar_mensagem(historico)
        historico.append({"role": "assistant", "content": resposta})

        print(f"\nPOLARIS: {resposta}\n")


# Inicia!
iniciar_chatbot()

  🚀  MISSION CONTROL AI — SOLARIS-1
  POLARIS — Artificial Response & Intelligence Assistant
  🟡 Bateria           : 26%
  ☀️  Geração Solar      : 85%
  🔴 Temp. Principal    : 100°C
  🟢 Temp. Propulsão    : 90°C
  🔴 Sinal              : PERDIDO
-------------------------------------------------------
  🚨 CRÍTICO: Sinal perdido — PROTOCOLO DE EMERGÊNCIA
-------------------------------------------------------
  Digite 'sair' para encerrar a sessão.

Você: sair

POLARIS: Sessão encerrada. Solaris-1, até a próxima janela de comunicação. 🚀

